In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Reuse the cleaned dataset if it exists.
# Otherwise, inspect the original dataframe.
import pandas as pd

clean_df = pd.read_parquet("../data/processed/np20ng_clean.parquet")

print(clean_df.shape)
clean_df.head()
if "clean_df" in globals():
    audit_df = clean_df.copy()
elif "df" in globals():
    audit_df = df.copy()
else:
    raise NameError(
        "Neither clean_df nor df exists. "
        "Load your saved dataset first."
    )

print("Dataset shape:", audit_df.shape)
print("\nColumns:")
print(audit_df.columns.tolist())

print("\nData types:")
print(audit_df.dtypes)

print("\nFirst 5 rows:")
display(audit_df.head())

(231186, 6)
Dataset shape: (231186, 6)

Columns:
['original_index', 'source', 'category', 'heading', 'content', 'normalized_content']

Data types:
original_index        int64
source                  str
category                str
heading                 str
content                 str
normalized_content      str
dtype: object

First 5 rows:


,original_index,source,category,heading,content,normalized_content
0,0,Gorkhapatra,Education,"अटेर गर्दै कतिपय विद्यालय, चैतमै भर्ना अभियान ...","काठमाडौं,चैत्र ७ गते । उपत्याकाका कतिपय विद्या...","काठमाडौं,चैत्र ७ गते । उपत्याकाका कतिपय विद्या..."
1,1,Gorkhapatra,Education,"निर्वाचन प्रचारमा विद्यालय, शिक्षक र विद्यार्...","प्रकाश सिलवाल काठमाडौँ, चैत ६ गते । विगतका निर...","प्रकाश सिलवाल काठमाडौँ, चैत ६ गते । विगतका निर..."
2,2,Gorkhapatra,Education,विद्यार्थीलाई परम्परागत सीप सिकाउँदै विद्यालय,"अमरराज नहर्की तनहुँ, चैत्र ५ गते । तनहुँको एक ...","अमरराज नहर्की तनहुँ, चैत्र ५ गते । तनहुँको एक ..."
3,3,Gorkhapatra,Education,आन्दोलन नगर्न शिक्षामन्त्रीको आग्रह,"गोरखापत्र समाचारदाता काठमाडौँ, चैत ३ गते । शिक...","गोरखापत्र समाचारदाता काठमाडौँ, चैत ३ गते । शिक..."
4,4,Gorkhapatra,Education,विश्वविद्यालयमा हडताल नगर्न शिक्षामन्त्रीको आग्रह,"काठमाडौं, चैत २ गते। शिक्षा विज्ञान तथा प्रविध...","काठमाडौं, चैत २ गते। शिक्षा विज्ञान तथा प्रविध..."


In [3]:
required_columns = ["content", "category", "source"]

missing_columns = [
    col for col in required_columns
    if col not in audit_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Missing values:")
print(audit_df[required_columns].isna().sum())

print("\nEmpty article texts:")
empty_text = (
    audit_df["content"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)
print(empty_text.sum())

print("\nNumber of categories:",
      audit_df["category"].nunique())

print("\nNumber of publishers:",
      audit_df["source"].nunique())

print("\nCategory distribution:")
display(
    audit_df["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="documents")
)

print("\nPublisher distribution — top 15:")
display(
    audit_df["source"]
    .value_counts()
    .head(15)
    .rename_axis("source")
    .reset_index(name="documents")
)

Missing values:
content     0
category    0
source      0
dtype: int64

Empty article texts:
0

Number of categories: 20

Number of publishers: 10

Category distribution:


,category,documents
0,Politics,31021
1,Society,27800
2,Sports,24392
3,Economy,23653
4,Entertainment,22664
5,World,17431
6,Business,11645
7,Bank,10526
8,Health,9469
9,Crime,8172



Publisher distribution — top 15:


,source,documents
0,Nepalkhabar,69504
1,Onlinekhabar,66716
2,Ekantipur,29875
3,Aarthiknews,25965
4,Setopati,15541
5,Nagarik,11106
6,Gorkhapatra,5059
7,Crimenews,3920
8,Nepalipatra,1952
9,Educationpati,1548


In [4]:
split_names = ["train_df", "val_df", "test_df"]

available_splits = {
    name: globals()[name]
    for name in split_names
    if name in globals()
}

for name, part in available_splits.items():
    print(f"{name}: {len(part):,} rows")

if len(available_splits) == 3:
    total_split_rows = sum(
        len(part) for part in available_splits.values()
    )

    print("\nTotal rows across splits:", total_split_rows)
    print("Rows in audit_df:", len(audit_df))

In [5]:
import re

def normalize_text(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text).lower()).strip()


clean_df["normalized_content"] = (
    clean_df["content"].map(normalize_text)
)

valid_text = clean_df["normalized_content"].ne("")

duplicate_mask = (
    clean_df.loc[valid_text, "normalized_content"]
    .duplicated(keep=False)
)

print(
    "Rows belonging to normalized duplicate groups:",
    duplicate_mask.sum()
)

duplicate_df = clean_df.loc[valid_text].loc[
    duplicate_mask
].copy()

duplicate_category_counts = (
    duplicate_df.groupby("normalized_content")["category"]
    .nunique()
)

print("\nDuplicate groups by number of categories:")
display(duplicate_category_counts.value_counts().sort_index())

conflicting_groups = duplicate_category_counts[
    duplicate_category_counts > 1
].index

display(
    duplicate_df[
        duplicate_df["normalized_content"].isin(conflicting_groups)
    ][["source", "category", "heading"]].head(20)
)

Rows belonging to normalized duplicate groups: 601

Duplicate groups by number of categories:


category
1    232
2     14
Name: count, dtype: int64

,source,category,heading
24795,Onlinekhabar,Bank,ट्राङ्लोबाट इसेवामनी ट्रान्सफरमार्फत नेपालमा र...
24873,Onlinekhabar,Bank,सानिमा बैंकका ग्राहकले शून्य प्रतिशत किस्तामा ...
27753,Nagarik,Economy,एनसेलको लोकल कल महसुल घट्यो
28277,Nagarik,Economy,सामसङले ल्यायो ग्यालेक्सी एस-२१ एफई ५जी
35643,Nepalipatra,Lifestyle,https://www.nepalipatra.comवास्तु टिप्स: जान्न...
35662,Nepalipatra,Lifestyle,https://www.nepalipatra.comवास्तुशास्त्र: पसल ...
35684,Nepalipatra,Lifestyle,https://www.nepalipatra.comवास्तु टिप्सः घरमा ...
35708,Nepalipatra,Lifestyle,https://www.nepalipatra.comब्लड प्रेसर लो भएमा...
35731,Nepalipatra,Lifestyle,https://www.nepalipatra.comवास्तुशास्त्र: घरको...
35797,Nepalipatra,Lifestyle,https://www.nepalipatra.comयस्ता व्यक्तिहरु सद...
